# 🧰 Voyager-Style Skill Library: Lifelong Learning via Accumulated Skills

## Learning Objectives
In this notebook, you will learn:
1. **Running generated code** - screen source with a denylist, then run it in a child process with a timeout
2. **A skill library** - store named, verified functions with usage counts
3. **Retrieval before generation** - reuse or compose skills before writing new code
4. **Verify before persisting** - check every new skill — and every reuse — against test cases that can fail
5. **A growing library** - watch four tasks generate, compose and build on skills

## Where this fits
`03_memgpt_tiered_memory.ipynb` managed *facts*. Voyager remembers *capabilities*: working code that later tasks can call or compose.

## Prerequisites
- No earlier notebook is required; Python functions and `subprocess` basics help
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Wang et al., *Voyager: An Open-Ended Embodied Agent with Large Language Models* (2023). [arXiv:2305.16291](https://arxiv.org/abs/2305.16291) |

## 📖 Definition

**Voyager** (Wang et al., 2023) was an embodied LLM agent that played Minecraft with no
fine-tuning at all. Its central trick was not a smarter prompt or a bigger model — it was
a **persistent skill library**: every time the agent solved a new sub-task, it saved the
working code as a reusable, named "skill." Future tasks first checked whether an existing
skill (or a composition of a few) already solved the problem before writing anything new.
Over time the agent built an ever-growing "automatic curriculum" of composable skills,
getting *faster and more capable* without ever touching the model's weights.

This notebook adapts that **core idea** to a domain-agnostic, code-executable setting —
small Python text/data-processing tasks — so no game engine or embodiment is required. The
mechanism transfers untouched:

1. Persistent, named skills (description + source code + usage count) stored in a library.
2. **Retrieval-before-generation**: an LLM call checks the library for a reusable or
   composable match before writing new code.
3. **Generate → run in a child process → verify → persist**: if nothing fits, the agent writes a
   new Python function, runs it in a child process with restricted builtins and a timeout, checks the result against test
   cases, and — only if it passes — commits it to the library for future reuse.
4. Demonstration across a short sequence of tasks of increasing complexity, where later
   tasks reuse or *compose* skills learned earlier, rather than re-deriving them.

### High-Level Workflow

```
                 ┌───────────────────────────┐
   new task ───▶ │ 1. Search skill library    │
                 │    (LLM decides: reuse /   │
                 │     compose / none fit)    │
                 └─────────────┬─────────────┘
                     match found│           no match
                                ▼                 ▼
                    ┌────────────────────┐  ┌─────────────────────────┐
                    │ Reuse existing      │  │ 2. Generate new Python   │
                    │ skill(s) directly   │  │    function via LLM      │
                    └─────────┬──────────┘  └────────────┬─────────────┘
                              │                            ▼
                              │              ┌──────────────────────────┐
                              │              │ 3. Run in a child         │
                              │              │    process: denylist,     │
                              │              │    builtins, timeout      │
                              │              └────────────┬─────────────┘
                              │                            ▼
                              │              ┌──────────────────────────┐
                              │              │ 4. Verify against         │
                              │              │    expected test cases    │
                              │              └────────────┬─────────────┘
                              │                    passes  │  fails
                              │                            ▼        ▼
                              │              ┌──────────────────┐ (retry /
                              │              │ 5. Persist to     │  give up)
                              │              │    skill library  │
                              │              └──────────────────┘
                              ▼
                    ┌────────────────────────────┐
                    │ Return result, bump         │
                    │ usage_count on skill(s)     │
                    └────────────────────────────┘
```

### When to Use

- Long-running or repeatedly-invoked agents that face many *similar-but-not-identical*
  sub-tasks over their lifetime (coding assistants, data-wrangling agents, game/simulation
  agents, workflow automators).
- Situations where fine-tuning is impractical/expensive, but you still want the agent to
  visibly "get better" at a domain over time.
- Any setting where code-as-skill is a natural unit of reusable capability (as opposed to
  free-text facts, which belong in the other memory architectures in this folder).

### Strengths & Weaknesses

**Strengths**

- **Lifelong learning without training**: capability compounds purely through accumulated,
  inspectable artifacts (functions), not gradient updates.
- **Composability**: skills call other skills, so complexity grows combinatorially, not
  linearly, with the number of stored skills.
- **Transparent & auditable**: every skill is literal, readable Python source — easy to
  review, test, or hand-edit, unlike opaque learned weights.
- **Cheap reuse**: once a skill exists, later invocations skip the (comparatively expensive)
  code-generation LLM call entirely.

**Weaknesses**

- **Retrieval quality bottleneck**: if the "does a skill already solve this?" lookup is
  wrong (misses a valid match, or wrongly reuses a bad match), errors compound silently.
- **Unbounded library growth**: without curation, the library accumulates near-duplicate or
  dead skills over a long run — real Voyager also needed periodic skill-library maintenance.
- **Sandboxing is essential and easy to under-engineer**: LLM-generated code executing
  locally is a real security surface. Part 1 shows what this notebook guards against, and what it doesn't.
- **No cross-task generalization inside a skill**: a skill only generalizes as far as its
  parameters allow; genuinely novel task shapes still require fresh code generation.

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: decides reuse vs. generation and writes new skills
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🛡️ Part 1: Running Generated Code — A Denylist and a Child Process

The whole point of this pattern is that the LLM writes Python at run time and we execute it. That code is
untrusted. Every skill — seeded or generated — goes through three layers:

- **A source denylist** rejects code containing obvious tokens (`import `, `__`, `open(`, `eval(`, `os.`,
  `subprocess`, …) before anything runs. It is a denylist, **not a sandbox**: Python offers attribute paths into
  the interpreter's internals that contain none of these tokens, so a determined author can get past it.
- **Restricted builtins**: the code runs with a hand-picked `__builtins__` (no `open`, `__import__`, `eval`,
  `exec`, `compile`, `input`).
- **A child process with a timeout**: every call runs in a fresh, isolated `python -I` process with an empty
  environment (so no API keys), killed after `SKILL_TIMEOUT_S` seconds. `while True: pass` cannot hang the
  notebook, and nothing the code does can change the notebook's own variables.

Nothing enters the library without passing its test cases. Treat this as a teaching harness for pure computation
(string, number and list processing): the child process still runs as you, with your files and network. Code from
a model you don't control belongs in a real isolated environment — a container, gVisor, a WASM runtime or a hosted
sandbox — with no network and no secrets.

In [ ]:
# ============================================================================
# PART 1: SCREEN THE SOURCE, THEN RUN IT IN A CHILD PROCESS WITH A TIMEOUT
# ============================================================================
import ast
import json
import re
import subprocess
import sys

SKILL_TIMEOUT_S = 5  # wall-clock limit for one skill call

FORBIDDEN_TOKENS = [
    "import ", "__", "open(", "exec(", "eval(", "compile(", "input(",
    "os.", "sys.", "subprocess", "socket", "requests", "shutil", "pathlib",
    "globals(", "locals(", "getattr(", "setattr(", "delattr(", "exit(", "quit(",
]

SAFE_BUILTINS = [
    "abs", "all", "any", "bool", "dict", "enumerate", "filter", "float",
    "int", "len", "list", "map", "max", "min", "range", "reversed",
    "round", "set", "sorted", "str", "sum", "tuple", "zip", "isinstance",
    "print",
]

# The child process defines every skill with restricted builtins, calls one, and prints the result as JSON
RUNNER = """
import builtins, json, sys
job = json.load(sys.stdin)
namespace = {"__builtins__": {name: getattr(builtins, name) for name in job["builtins"]}}
for source in job["sources"]:
    exec(source, namespace)
print(json.dumps(namespace[job["function"]](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/04_agentic_memory_architectures/notebooks/*job["args"])))
"""


def screen_source(source_code: str) -> None:
    """Static denylist screen. Raises ValueError if the source contains a forbidden token."""
    lowered = source_code.lower()
    for token in FORBIDDEN_TOKENS:
        if token in lowered:
            raise ValueError(f"Rejected generated code: forbidden token '{token}' found.")


def run_skill(sources: list[str], function_name: str, args: tuple, timeout: float = SKILL_TIMEOUT_S):
    """Run `function_name(*args)` in a fresh Python process that sees only `sources` and SAFE_BUILTINS."""
    job = json.dumps({"builtins": SAFE_BUILTINS, "sources": sources, "function": function_name, "args": list(args)})
    try:
        done = subprocess.run([sys.executable, "-I", "-c", RUNNER], input=job, capture_output=True,
                              text=True, timeout=timeout, env={})  # -I and env={}: no user site, no API keys
    except subprocess.TimeoutExpired:
        raise TimeoutError(f"{function_name} ran longer than {timeout}s and was killed") from None
    if done.returncode != 0:
        raise RuntimeError((done.stderr.strip().splitlines() or ["the child process failed"])[-1])
    return json.loads(done.stdout.strip().splitlines()[-1])  # the last line; a skill may print before it


def load_skill(source_code: str, function_name: str, helper_sources: tuple[str, ...] = ()):
    """Screen `source_code`, check it defines `function_name`, and return a callable that runs it in a child process.

    `helper_sources` are verified skills the new one may call (composition)."""
    screen_source(source_code)
    try:
        tree = ast.parse(source_code)  # parsing runs nothing
    except SyntaxError as exc:
        raise ValueError(f"Generated code is not valid Python: {exc}") from None
    if not any(isinstance(node, ast.FunctionDef) and node.name == function_name for node in tree.body):
        raise ValueError(f"Generated code did not define expected function '{function_name}'.")
    sources = [*helper_sources, source_code]
    return lambda *args: run_skill(sources, function_name, args)


print(f"✅ Denylist of {len(FORBIDDEN_TOKENS)} tokens, {len(SAFE_BUILTINS)} builtins, {SKILL_TIMEOUT_S}s per call")

In [ ]:
# ============================================================================
# PART 1 IN ACTION: A REJECTED SOURCE AND A RUNAWAY LOOP
# ============================================================================
try:
    load_skill("import os\ndef leak(text):\n    return dict(os.environ)", "leak")
except ValueError as error:
    print(f"Denylist: {error}")

spin_source = "def spin(text):\n    while True:\n        pass"
screen_source(spin_source)  # passes: an infinite loop contains no forbidden token
try:
    run_skill([spin_source], "spin", ("x",), timeout=2)
except TimeoutError as error:
    print(f"Timeout:  {error}")

The denylist stopped the `import` before anything ran. The infinite loop contains no forbidden token, so only the
timeout could stop it: the child process was killed after 2 seconds and the notebook carried on.

---
## 📚 Part 2: The Skill Library

The library (`SKILL_LIBRARY`) is a dict of `skill_name -> {description, code, function_name, function, usage_count}`.
We seed it with two trivial skills, `count_vowels` and `reverse_string` — analogous to Voyager's initial primitive
actions. Every skill's `function` comes from `load_skill`, so seeds and generated skills get the same screening
and the same child process.

In [ ]:
# ============================================================================
# PART 2: THE SKILL LIBRARY, SEEDED WITH TWO PRIMITIVES
# ============================================================================
SKILL_LIBRARY: dict = {}


def _register(name: str, description: str, source_code: str, function_name: str) -> None:
    """Screen a skill's source and register it with a callable that runs it in a child process."""
    func = load_skill(source_code, function_name)
    SKILL_LIBRARY[name] = {
        "description": description,
        "code": source_code,
        "function_name": function_name,
        "function": func,
        "usage_count": 0,
    }


# --- Seed skills (Voyager's "primitive actions") ---------------------------
_register(
    name="count_vowels",
    description="Count the number of vowels (a, e, i, o, u, case-insensitive) in a string.",
    source_code=(
        "def count_vowels(text):\n"
        "    return sum(1 for ch in text.lower() if ch in 'aeiou')"
    ),
    function_name="count_vowels",
)

_register(
    name="reverse_string",
    description="Reverse a string.",
    source_code=(
        "def reverse_string(text):\n"
        "    return text[::-1]"
    ),
    function_name="reverse_string",
)


def list_skills() -> str:
    lines = []
    for name, skill in SKILL_LIBRARY.items():
        lines.append(
            f"- {name}: {skill['description']} (used {skill['usage_count']}x)"
        )
    return "\n".join(lines)


print("Seeded skill library:")
print(list_skills())

---
## 🔍 Part 3: Retrieval Before Generation

Before writing anything new, we ask the LLM to look at the current library's descriptions
and decide: does an existing skill (or combination of skills) already solve this task? This
mirrors Voyager's skill-retrieval step, done here via a direct LLM judgment call rather than
a vector index — simple, and sufficient for a library of this size.

The LLM must answer in a strict, parseable format so we can act on it programmatically.

In [ ]:
# ============================================================================
# PART 3: ASK THE LLM WHETHER THE LIBRARY ALREADY SOLVES A TASK
# ============================================================================
import json as _json


def find_matching_skill(task_description: str) -> dict:
    """
    Ask the LLM whether the existing skill library already solves `task_description`.
    Returns a dict: {"reuse": bool, "skill_names": [...], "reasoning": str}
    """
    prompt = f'''You are maintaining a library of reusable Python skills for an agent.

Current skill library:
{list_skills()}

New task: "{task_description}"

Decide whether the existing skills already fully solve this task, either by using ONE
skill directly, or by COMPOSING two or more existing skills in sequence. Do NOT propose
reuse if the task needs logic that no combination of existing skills provides.

Respond with ONLY a JSON object, no prose, no markdown fences, in this exact shape:
{{"reuse": true or false, "skill_names": ["skill_a", "skill_b"], "reasoning": "..."}}

If reuse is false, "skill_names" must be an empty list.'''

    response = llm.invoke(prompt).content.strip()
    # Be defensive in case the model wraps the JSON in a code fence anyway.
    response = response.strip("`")
    if response.lower().startswith("json"):
        response = response[4:].strip()
    try:
        return _json.loads(response)
    except _json.JSONDecodeError:
        return {"reuse": False, "skill_names": [], "reasoning": "unparseable response"}


print("✅ Skill lookup ready")

### 3.1 ✍️ Skill Generation (When Nothing Fits)

When no existing skill covers the task, the LLM writes a brand-new, single, pure Python
function. The prompt explicitly tells it:

- the exact function name to use,
- that it may call any already-verified skill functions by name (their signatures are
  listed) if composition helps,
- that it must not use `import`, file/network I/O, or any of the other denylisted patterns
  from Part 1 (kept simple: pure computation only).

In [ ]:
# ============================================================================
# PART 3.1: WRITE A NEW SKILL WHEN NOTHING FITS
# ============================================================================
def _slugify(task_description: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", task_description.lower()).strip("_")
    return slug[:40] or "new_skill"


def generate_new_skill(task_description: str) -> tuple[str, str, str]:
    """
    Ask the LLM to write a new Python function solving `task_description`.
    Returns (function_name, source_code, description).
    """
    function_name = f"skill_{_slugify(task_description)}"
    available = "\n".join(
        f"- {name}({skill['function_name']}): {skill['description']}"
        for name, skill in SKILL_LIBRARY.items()
    )

    prompt = f'''Write a single pure Python function named exactly `{function_name}` that
solves this task:

"{task_description}"

You may call any of these already-available helper functions by name if it helps
(they are already defined in the execution namespace, do not redefine them):
{available}

Strict rules:
- Output ONLY the function definition, no explanation, no markdown code fences.
- Do not use `import`, file I/O, network calls, `eval`, `exec`, or any dunder attributes.
- The function must be deterministic, pure computation only, and self-contained aside from
  the helper functions listed above.
'''

    raw = llm.invoke(prompt).content.strip()
    # Strip a markdown fence if the model adds one despite instructions.
    raw = re.sub(r"^```(python)?", "", raw.strip(), flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw.strip()).strip()

    description = f"Auto-generated skill for task: {task_description}"
    return function_name, raw, description


print("✅ Skill generator ready")

---
## ✅ Part 4: Verify Before Persisting

`verify_skill` runs a candidate on every test case and fails on the first wrong result, error or timeout. Test
cases only protect the library if they *can* fail: when every expected answer is `True`, a function that always
returns `True` passes, and the library stores a skill that checks nothing.

In [ ]:
# ============================================================================
# PART 4: RUN A CANDIDATE AGAINST TEST CASES
# ============================================================================
def verify_skill(func, test_cases: list[tuple]) -> bool:
    """test_cases: list of (args_tuple, expected_output). Returns True iff all pass."""
    for args, expected in test_cases:
        try:
            actual = func(*args)
        except Exception as exc:  # noqa: BLE001 -- we want to catch and report any failure
            print(f"  verification error on input {args}: {exc}")
            return False
        if actual != expected:
            print(f"  verification failed on input {args}: got {actual!r}, expected {expected!r}")
            return False
    return True


print("✅ Verifier ready")

### 4.1 🔗 The `solve_task` Orchestrator

This ties retrieval, generation, execution in a child process, verification, and persistence
together into the Voyager lifecycle described in the workflow diagram above.

In [ ]:
# ============================================================================
# PART 4.1: THE VOYAGER LIFECYCLE FOR ONE TASK
# ============================================================================
def _reused_result(skill_names: list[str], task_args: tuple):
    """Apply the reused skill(s) to task_args. Single skill: call directly.
    Multiple skills: compose left-to-right, feeding each skill's output into the next."""
    value = task_args[0]
    for name in skill_names:
        skill = SKILL_LIBRARY[name]
        value = skill["function"](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/04_agentic_memory_architectures/notebooks/value)
        skill["usage_count"] += 1
    return value


def solve_task(task_description: str, task_args: tuple, test_cases: list[tuple]):
    """
    Full Voyager-style lifecycle for one task:
      1. Try to reuse/compose existing skills (LLM lookup).
      2. If nothing fits, generate + run in a child process + verify + persist a new skill.
      3. Return the result of applying the (reused or newly persisted) skill to task_args.
    """
    print(f"\n=== Task: {task_description} ===")
    lookup = find_matching_skill(task_description)
    print(f"Lookup decision: {lookup}")

    if lookup.get("reuse") and lookup.get("skill_names"):
        skill_names = [n for n in lookup["skill_names"] if n in SKILL_LIBRARY]
        if skill_names:
            def composed(text):  # the composition, without bumping usage counts
                for n in skill_names:
                    text = SKILL_LIBRARY[n]["function"](https://github.com/Sourav692/ai-engineering-demystified-08-advanced-agent-systems/tree/main/modules/04_agentic_memory_architectures/notebooks/text)
                return text

            if test_cases and not verify_skill(composed, test_cases):
                print("Reused composition failed the test cases -- falling back to generation.")
            else:
                result = _reused_result(skill_names, task_args)
                print(f"Reused skill(s) {skill_names} -> result = {result!r}")
                return result
        else:
            print("Lookup named unknown skills; falling back to generation.")

    print("No existing skill fits -- generating a new one.")
    function_name, source_code, description = generate_new_skill(task_description)
    print(f"Generated candidate `{function_name}`:\n{source_code}\n")

    helper_sources = tuple(s["code"] for s in SKILL_LIBRARY.values())  # the new skill may call any of these
    try:
        func = load_skill(source_code, function_name, helper_sources)
    except ValueError as exc:
        print(f"Rejected before running: {exc}")
        return None

    if not test_cases:
        print("No test cases -- refusing to persist an unverified skill.")
        return None

    if not verify_skill(func, test_cases):
        print("New skill failed verification -- NOT persisted.")
        return None

    SKILL_LIBRARY[function_name] = {
        "description": description,
        "code": source_code,
        "function_name": function_name,
        "function": func,
        "usage_count": 1,
    }
    result = func(*task_args)
    calls = [name for name in SKILL_LIBRARY if name != function_name and re.search(rf"\b{name}\(", source_code)]
    print(f"Verified and persisted new skill `{function_name}` (calls library skills: {calls or 'none'}) "
          f"-> result = {result!r}")
    return result


print("✅ Orchestrator ready")

---
## 🧪 Part 5: The Lifecycle Across Four Tasks

- **Task 1** and **Task 2** need logic the two seeds don't have (counting uppercase letters, removing whitespace),
  so the lookup should say `reuse: false`, and the agent generates, verifies and persists two new skills.
- **Task 3** is phrased so that it can be solved by **composing** the skill generated in Task 2 with the
  `reverse_string` seed — no new code at all.
- **Task 4** (a palindrome check that ignores case and spaces) needs new logic, but the new function can *call*
  skills already in the library. Its test cases include both `True` and `False` answers, so a wrong function fails.

Each task prints the lookup decision so you can see when the agent reuses vs. generates.

In [ ]:
# ============================================================================
# TASK 1: A NEW SKILL
# ============================================================================
# Task 1: a new phrasing of a string-processing task not already in the library.
result_1 = solve_task(
    task_description="Count how many uppercase letters are in a string",
    task_args=("Hello World AI",),
    test_cases=[
        (("Hello World AI",), 4),
        (("abc",), 0),
        (("XYZ",), 3),
    ],
)
result_1

### Discussion of the Output

The library had no skill for counting uppercase letters, so the lookup step correctly
returns `reuse: false`, the LLM writes a new `skill_count_how_many_uppercase_letters_..._`
function, it passes the denylist and all three test cases, and is persisted. The library now
has three skills.

In [ ]:
# ============================================================================
# TASK 2: ANOTHER NEW SKILL
# ============================================================================
# Task 2: another new, distinct skill.
result_2 = solve_task(
    task_description="Remove all whitespace from a string",
    task_args=("a b  c   d",),
    test_cases=[
        (("a b  c   d",), "abcd"),
        (("no spaces",), "nospaces"),
        (("",), ""),
    ],
)
result_2

### Discussion of the Output

Again no existing skill matches, so a fourth skill is generated, verified, and persisted.
At this point the library holds four independent skills: `count_vowels`, `reverse_string`,
and the two just generated.

In [ ]:
# ============================================================================
# TASK 3: COMPOSE A GENERATED SKILL WITH A SEED
# ============================================================================
result_3 = solve_task(
    task_description="Remove all whitespace from a string and then reverse the result",
    task_args=("Voy ager",),
    test_cases=[(("Voy ager",), "regayoV"), (("a b c",), "cba"), (("",), "")],  # checked on the reuse path too
)
generated_reused = [name for name, s in SKILL_LIBRARY.items() if name.startswith("skill_") and s["usage_count"] > 1]
print(f"{'✅' if generated_reused else '❌'} a skill the agent generated earlier was reused: {generated_reused or 'none'}")
result_3

### Discussion of the Output

This is the key Voyager moment: the lookup recognises that the whitespace remover generated in Task 2, followed
by `reverse_string`, already solves the task. It returns `reuse: true` with both names, and the orchestrator
composes them directly — **no code-generation call is made at all**. A skill the agent wrote itself is now a
building block. The composition is still checked against the test cases before its result is used, and both
skills' `usage_count` increments, visible in the final library listing below.

In [ ]:
# ============================================================================
# TASK 4: NEW LOGIC THAT CAN CALL EXISTING SKILLS
# ============================================================================
palindrome_tests = [
    (("Never odd or even",), True),
    (("Voyager",), False),
    (("Step on no pets",), True),
    (("ab",), False),
]
result_4 = solve_task(
    task_description="Check whether a string is a palindrome, ignoring letter case and whitespace -- return True or False",
    task_args=("Never odd or even",),
    test_cases=palindrome_tests,
)
result_4

### Discussion of the Output

Task 4 cannot be solved by *pure* reuse (the library has no comparison skill), so the lookup falls through to
generation. The generation prompt exposes the existing skills as callables the new function may use, so the model
can write something like:

```python
def skill_check_whether_a_string_is_a_palindrome_i(text):
    cleaned = skill_remove_all_whitespace_from_a_string(text).lower()
    return cleaned == reverse_string(cleaned)
```

The "calls library skills" part of the output shows which skills the generated code actually calls — composition
at the code-generation level, not just at the orchestration level. The test cases include `False` answers, so a
function that always returns `True` would be rejected; the next cell shows exactly that.

### 5.1 ❌ A Wrong Candidate Is Caught

Here is a candidate that ignores its input and always answers `True`. Against Task 4's test cases it fails on
the first `False` case, so `solve_task` would never persist it.

In [ ]:
# ============================================================================
# A WRONG CANDIDATE FAILS VERIFICATION
# ============================================================================
always_true = load_skill("def always_true(text):\n    return True", "always_true")
passed = verify_skill(always_true, palindrome_tests)
print(f"{'✅' if not passed else '❌'} the constant-True candidate was rejected, so it would not be persisted")

In [ ]:
# ============================================================================
# THE LIBRARY AFTER FOUR TASKS
# ============================================================================
print("Final skill library state:\n")
print(list_skills())
print(f"\nTotal skills accumulated: {len(SKILL_LIBRARY)}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Running generated code
- A source denylist, restricted builtins and a child process with a timeout stop a runaway or careless skill from hanging the notebook or changing its state — but a denylist is not a sandbox: untrusted code belongs in a real isolated environment

### 2. A skill library
- Named, verified functions with usage counts: accumulated, inspectable artifacts instead of weight updates

### 3. Retrieval before generation
- An LLM lookup decides whether existing skills — alone or composed — already solve a task; a reused skill skips the code-generation call entirely

### 4. Verify before persisting
- Every new skill and every reuse is checked against test cases, and the test cases must be able to fail: a constant-`True` candidate was rejected

### 5. A growing library
- Task 3 composed a skill the agent had generated itself; Task 4's new function could call library skills inside its own code — each solved task makes the next one cheaper

### Next Steps
- **`05_agent_workflow_memory.ipynb`**: remember successful *tool-call procedures* and adapt them to new tasks